In [33]:
###TASK_1
### Import libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd

books = []

for page in range(1, 6):

    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    for book in soup.find_all("article", class_="product_pod"):

        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        star_rating = book.p["class"][1]
        availability = book.find("p", class_="instock availability").text.strip()

 ### Open individual book page
        href = book.h3.a["href"]
        book_url = "https://books.toscrape.com/catalogue/" + href

        book_response = requests.get(book_url)
        book_soup = BeautifulSoup(book_response.text, "html.parser")

  ### Get category
        breadcrumb = book_soup.find("ul", class_="breadcrumb")
        category = breadcrumb.find_all("li")[2].text.strip()

        books.append({
            "title": title,
            "price": price,
            "star_rating": star_rating,
            "availability": availability,
            "category": category
        })

df = pd.DataFrame(books)

print(df.head())

df.to_csv("books.csv", index=False)

                                   title    price star_rating availability  \
0                   A Light in the Attic  Â£51.77       Three     In stock   
1                     Tipping the Velvet  Â£53.74         One     In stock   
2                             Soumission  Â£50.10         One     In stock   
3                          Sharp Objects  Â£47.82        Four     In stock   
4  Sapiens: A Brief History of Humankind  Â£54.23        Five     In stock   

             category  
0              Poetry  
1  Historical Fiction  
2             Fiction  
3             Mystery  
4             History  


In [34]:
df

,title,price,star_rating,availability,category
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction
2,Soumission,Â£50.10,One,In stock,Fiction
3,Sharp Objects,Â£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History
...,...,...,...,...,...
95,Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...,Â£19.92,Two,In stock,Sequential Art
96,"Layered: Baking, Building, and Styling Spectac...",Â£40.11,One,In stock,Food and Drink
97,Judo: Seven Steps to Black Belt (an Introducto...,Â£53.90,Two,In stock,Add a comment
98,Join,Â£35.67,Five,In stock,Science Fiction


In [35]:
##TASK_2
###load scraped data
df=pd.read_csv("books.csv")
df

,title,price,star_rating,availability,category
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction
2,Soumission,Â£50.10,One,In stock,Fiction
3,Sharp Objects,Â£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History
...,...,...,...,...,...
95,Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...,Â£19.92,Two,In stock,Sequential Art
96,"Layered: Baking, Building, and Styling Spectac...",Â£40.11,One,In stock,Food and Drink
97,Judo: Seven Steps to Black Belt (an Introducto...,Â£53.90,Two,In stock,Add a comment
98,Join,Â£35.67,Five,In stock,Science Fiction


In [36]:
## Remove currency symbol and convert to float
df["price_gbp"]=df["price"].str.replace("Â£","",regex=False).astype(float)




In [37]:
df

,title,price,star_rating,availability,category,price_gbp
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry,51.77
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction,53.74
2,Soumission,Â£50.10,One,In stock,Fiction,50.10
3,Sharp Objects,Â£47.82,Four,In stock,Mystery,47.82
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History,54.23
...,...,...,...,...,...,...
95,Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...,Â£19.92,Two,In stock,Sequential Art,19.92
96,"Layered: Baking, Building, and Styling Spectac...",Â£40.11,One,In stock,Food and Drink,40.11
97,Judo: Seven Steps to Black Belt (an Introducto...,Â£53.90,Two,In stock,Add a comment,53.90
98,Join,Â£35.67,Five,In stock,Science Fiction,35.67


In [38]:
###convert rating text to integer
rating_map={"One":1,
            "Two":2,
            "Three":3,
            "Four":4,
            "Five":5}


df["star_rating"]=df["star_rating"].map(rating_map)

In [39]:
df['star_rating']

,star_rating
0,3
1,1
2,1
3,4
4,5
...,...
95,2
96,1
97,2
98,5


In [40]:
###Convert availability to boolean
df["in_stock"]=df["availability"].str.contains("In stock")


In [41]:
df["in_stock"]

,in_stock
0,True
1,True
2,True
3,True
4,True
...,...
95,True
96,True
97,True
98,True


In [42]:
###TASK_3
###Convert GBP to INR
GBP_TO_INR=105.50
df["price_inr"]=df["price_gbp"]*GBP_TO_INR

In [43]:
df["price_inr"]

,price_inr
0,5461.735
1,5669.570
2,5285.550
3,5045.010
4,5721.265
...,...
95,2101.560
96,4231.605
97,5686.450
98,3763.185


In [44]:
###Handling missing values
df["price_gbp"].fillna(df["price_gbp"].median(),inplace=True)
df["star_rating"].fillna(df["star_rating"].median(),inplace=True)

/tmp/ipykernel_1069/845160892.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["price_gbp"].fillna(df["price_gbp"].median(),inplace=True)


In [45]:
df['price_gbp']
df['star_rating']

,star_rating
0,3
1,1
2,1
3,4
4,5
...,...
95,2
96,1
97,2
98,5


In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         100 non-null    object 
 1   price         100 non-null    object 
 2   star_rating   100 non-null    int64  
 3   availability  100 non-null    object 
 4   category      100 non-null    object 
 5   price_gbp     100 non-null    float64
 6   in_stock      100 non-null    bool   
 7   price_inr     100 non-null    float64
dtypes: bool(1), float64(2), int64(1), object(4)
memory usage: 5.7+ KB


In [47]:
###save clean data
df.to_csv("clean_books.csv",index=False)

In [48]:
print(df.head())

                                   title    price  star_rating availability  \
0                   A Light in the Attic  Â£51.77            3     In stock   
1                     Tipping the Velvet  Â£53.74            1     In stock   
2                             Soumission  Â£50.10            1     In stock   
3                          Sharp Objects  Â£47.82            4     In stock   
4  Sapiens: A Brief History of Humankind  Â£54.23            5     In stock   

             category  price_gbp  in_stock  price_inr  
0              Poetry      51.77      True   5461.735  
1  Historical Fiction      53.74      True   5669.570  
2             Fiction      50.10      True   5285.550  
3             Mystery      47.82      True   5045.010  
4             History      54.23      True   5721.265  


In [49]:
###TASK_4
##import  libraries
import sqlite3
import pandas as pd
###load cleaned data
df = pd.read_csv("/content/clean_books.csv")
###Database connection
conn = sqlite3.connect("books.db")
cursor = conn.cursor()

# Create categories table
cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE
)
""")






In [50]:
# Create books table
cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY(category_id) REFERENCES categories(category_id)
)
""")


In [51]:
print(df.columns)

Index(['title', 'price', 'star_rating', 'availability', 'category',
       'price_gbp', 'in_stock', 'price_inr'],
      dtype='object')


In [52]:
print(df.head())

                                   title    price  star_rating availability  \
0                   A Light in the Attic  Â£51.77            3     In stock   
1                     Tipping the Velvet  Â£53.74            1     In stock   
2                             Soumission  Â£50.10            1     In stock   
3                          Sharp Objects  Â£47.82            4     In stock   
4  Sapiens: A Brief History of Humankind  Â£54.23            5     In stock   

             category  price_gbp  in_stock  price_inr  
0              Poetry      51.77      True   5461.735  
1  Historical Fiction      53.74      True   5669.570  
2             Fiction      50.10      True   5285.550  
3             Mystery      47.82      True   5045.010  
4             History      54.23      True   5721.265  


In [53]:
# Insert categories
categories = df["category"].unique()

for category in categories:
    cursor.execute(
        "INSERT OR IGNORE INTO categories(category_name) VALUES(?)",
        (category,)
    )

In [54]:
# Insert books
for _, row in df.iterrows():
    cursor.execute(
        "SELECT category_id FROM categories WHERE category_name=?",
        (row["category"],)
    )
    category_id = cursor.fetchone()[0]

    cursor.execute("""
    INSERT INTO books
    (title, price_gbp, price_inr, rating, in_stock, category_id)
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        row["star_rating"],
        int(row["in_stock"]),
        category_id
    ))

conn.commit()
conn.close()

print("Database created successfully!")

Database created successfully!


In [55]:
##TASK_5
###Execute 5 sql queries
import sqlite3
import pandas as pd
###Database connection
conn=sqlite3.connect("books.db")
cursor=conn.cursor()


In [56]:
###QUERY-1 SELECT
query1="SELECT * FROM books LIMIT 10"
print(pd.read_sql(query1,conn))

   book_id                                              title  price_gbp  \
0        1                               A Light in the Attic      51.77   
1        2                                 Tipping the Velvet      53.74   
2        3                                         Soumission      50.10   
3        4                                      Sharp Objects      47.82   
4        5              Sapiens: A Brief History of Humankind      54.23   
5        6                                    The Requiem Red      22.65   
6        7  The Dirty Little Secrets of Getting Your Dream...      33.34   
7        8  The Coming Woman: A Novel Based on the Life of...      17.93   
8        9  The Boys in the Boat: Nine Americans and Their...      22.60   
9       10                                    The Black Maria      52.15   

   price_inr  rating  in_stock  category_id  
0   5461.735       3         1            1  
1   5669.570       1         1            2  
2   5285.550       1     

In [57]:
###QUERY2_WHERE
query2="SELECT title rating FROM books WHERE rating=5"
print(pd,pd.read_sql(query2,conn))

<module 'pandas' from '/usr/local/lib/python3.12/dist-packages/pandas/__init__.py'>                                                rating
0               Sapiens: A Brief History of Humankind
1                                         Set Me Free
2   Scott Pilgrim's Precious Little Life (Scott Pi...
3                           Rip it Up and Start Again
4                          Chase Me (Paris Nights #2)
5                                          Black Dust
6   Worlds Elsewhere: Journeys Around Shakespeareâ...
7   The Four Agreements: A Practical Guide to Pers...
8                                   The Elephant Tree
9                                      Sophie's World
10                        Private Paris (Private #10)
11  #HigherSelfie: Wake Up Your Life. Free Your So...
12                       We Love You, Charlie Freeman
13                                             Thirst
14  The Inefficiency Assassin: Time Management Tac...
15  The Activist's Tao Te Ching: Ancient Advice fo..

In [58]:
##QUERY3_ORDERBY+LIMIT
query3="""
SELECT title, price_inr FROM books ORDER BY price_inr DESC LIMIT 10"""
print(pd.read_sql(query3,conn))

                                               title  price_inr
0       The Death of Humanity: and the Case for Life   6130.605
1       The Death of Humanity: and the Case for Life   6130.605
2                     Slow States of Collapse: Poems   6046.205
3                     Slow States of Collapse: Poems   6046.205
4  Our Band Could Be Your Life: Scenes from the A...   6039.875
5  Our Band Could Be Your Life: Scenes from the A...   6039.875
6                                The Past Never Ends   5960.750
7                                The Past Never Ends   5960.750
8  The Pioneer Woman Cooks: Dinnertime: Comfort C...   5951.255
9  The Pioneer Woman Cooks: Dinnertime: Comfort C...   5951.255


In [59]:
##QUERY4_DISTINCT
query4= "SELECT DISTINCT category_name FROM categories"
print(pd.read_sql(query4,conn))

         category_name
0        Add a comment
1                  Art
2             Business
3            Childrens
4         Contemporary
5              Default
6              Fantasy
7              Fiction
8       Food and Drink
9               Health
10  Historical Fiction
11             History
12              Horror
13               Music
14             Mystery
15           New Adult
16          Nonfiction
17          Philosophy
18              Poetry
19            Politics
20             Romance
21             Science
22     Science Fiction
23           Self Help
24      Sequential Art
25        Spirituality
26            Thriller
27              Travel
28         Young Adult


In [60]:
##QUERY5_JOIN
query5="""
SELECT b.title,
        c.category_name,
        b.rating,
        b.price_inr
 FROM books b
 JOIN categories c
 ON b.category_id=c.category_id
 """
print(pd. read_sql(query5,conn))

                                                 title       category_name  \
0                                 A Light in the Attic              Poetry   
1                                   Tipping the Velvet  Historical Fiction   
2                                           Soumission             Fiction   
3                                        Sharp Objects             Mystery   
4                Sapiens: A Brief History of Humankind             History   
..                                                 ...                 ...   
195  Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...      Sequential Art   
196  Layered: Baking, Building, and Styling Spectac...      Food and Drink   
197  Judo: Seven Steps to Black Belt (an Introducto...       Add a comment   
198                                               Join     Science Fiction   
199          In the Country We Love: My Family Divided          Nonfiction   

     rating  price_inr  
0         3   5461.735  
1         1  

In [61]:
##TASK_6
###Read two sql query result into DataFrames
df1=pd.read_sql(query2,conn)
print(df1.head())
df2=pd.read_sql(query5,conn)
print(df2.head())

                                              rating
0              Sapiens: A Brief History of Humankind
1                                        Set Me Free
2  Scott Pilgrim's Precious Little Life (Scott Pi...
3                          Rip it Up and Start Again
4                         Chase Me (Paris Nights #2)
                                   title       category_name  rating  \
0                   A Light in the Attic              Poetry       3   
1                     Tipping the Velvet  Historical Fiction       1   
2                             Soumission             Fiction       1   
3                          Sharp Objects             Mystery       4   
4  Sapiens: A Brief History of Humankind             History       5   

   price_inr  
0   5461.735  
1   5669.570  
2   5285.550  
3   5045.010  
4   5721.265  


In [62]:
###Reproduce the JOIN using pd.merge()
books_df=pd.read_sql("SELECT * FROM books",conn)
categories_df=pd.read_sql("SELECT * FROM categories",conn)

merged_df=pd.merge(books_df,
                   categories_df,
                   on="category_id",
                   how="inner")
merged_df=merged_df[["title","category_name"]]
print(merged_df.head())


                                   title       category_name
0                   A Light in the Attic              Poetry
1                     Tipping the Velvet  Historical Fiction
2                             Soumission             Fiction
3                          Sharp Objects             Mystery
4  Sapiens: A Brief History of Humankind             History


In [63]:
##Compare SQL JOIN with pandas merge.

sql_df = pd.read_sql("""
SELECT b.title, c.category_name
FROM books b
JOIN categories c
ON b.category_id = c.category_id
""", conn)
print(sql_df.head())
print(merged_df.head())

                                   title       category_name
0                   A Light in the Attic              Poetry
1                     Tipping the Velvet  Historical Fiction
2                             Soumission             Fiction
3                          Sharp Objects             Mystery
4  Sapiens: A Brief History of Humankind             History
                                   title       category_name
0                   A Light in the Attic              Poetry
1                     Tipping the Velvet  Historical Fiction
2                             Soumission             Fiction
3                          Sharp Objects             Mystery
4  Sapiens: A Brief History of Humankind             History


In [64]:
###Close database
conn.close()